[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/03_Training_Strategies/05_scaling_laws/05_scaling_laws.ipynb)

# 05. Scaling Laws for Multimodal Training

**This notebook covers:**
- Chinchilla scaling law formulas and plots
- Compute-optimal training calculator
- Data scaling vs model scaling analysis

**Runtime:** ~10–15 minutes on CPU

---

> **Theory & derivations:** See [README.md](./README.md) for full step-by-step math.


In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("Repository already cloned")

    print("Installing dependencies...")
    !pip install -q -r {REPO_DIR}/requirements.txt

    MODULE_DIR = f"{REPO_DIR}/03_Training_Strategies/05_scaling_laws"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Colab setup complete — {os.getcwd()}")

    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

try:
    from utils.visualization import set_style
    from utils.helpers import count_parameters, get_device
    set_style()
except ImportError:
    def set_style():
        plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total:,}")
        return total
    def get_device():
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_style()

torch.manual_seed(42)
np.random.seed(42)
device = get_device() if callable(get_device) else torch.device('cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

## 1. Chinchilla Scaling Laws

Hoffmann et al. (2022): optimal tokens $D^* \approx 20 N$ for parameter count $N$.

$$L(N, D) \approx E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$$

Empirically $\alpha \approx 0.34$, $\beta \approx 0.28$.


In [ ]:
def chinchilla_loss(N, D, E=1.69, A=406.4, B=410.7, alpha=0.34, beta=0.28):
    return E + A / (N ** alpha) + B / (D ** beta)

N_vals = np.logspace(6, 10, 50)  # 1M to 10B params
D_vals = 20 * N_vals
L_vals = chinchilla_loss(N_vals, D_vals)

plt.figure(figsize=(8, 4))
plt.loglog(N_vals, L_vals, label='L(N, D=20N)')
plt.xlabel('Parameters N'); plt.ylabel('Loss L'); plt.title('Chinchilla IsoFLOP Curve')
plt.grid(True, which='both', alpha=0.3); plt.legend(); plt.show()

## 2. Compute-Optimal Training Calculator

Compute $\approx 6 N D$ FLOPs (forward + backward factor).


In [ ]:
def compute_optimal_tokens(N, ratio=20):
    return ratio * N

def training_flops(N, D):
    return 6 * N * D

def estimate_days(flops, gpu_tflops=312, n_gpus=8, utilization=0.45):
    seconds = flops / (gpu_tflops * 1e12 * n_gpus * utilization)
    return seconds / 86400

configs = [
    ("Mini-CLIP", 1.5e6, 30e6),
    ("CLIP-Base", 150e6, 3e9),
    ("CLIP-Large", 430e6, 8.6e9),
    ("LLaVA-7B stage1", 7e9, 140e9),
]

print(f"{'Model':<20} {'N':>12} {'D (tokens)':>14} {'FLOPs':>12} {'GPU-days (8xA100)':>18}")
print('-' * 82)
for name, N, D in configs:
    flops = training_flops(N, D)
    days = estimate_days(flops)
    print(f"{name:<20} {N:12.2e} {D:14.2e} {flops:12.2e} {days:18.1f}")

## 3. Data Scaling vs Model Scaling


In [ ]:
N_fixed = 1e9
D_range = np.logspace(7, 11, 40)
L_data = chinchilla_loss(N_fixed, D_range)

D_fixed = 20e9
N_range = np.logspace(7, 10, 40)
L_model = chinchilla_loss(N_range, D_fixed)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].loglog(D_range, L_data); ax[0].set_xlabel('Tokens D'); ax[0].set_title('Fix N=1B, vary data')
ax[1].loglog(N_range, L_model); ax[1].set_xlabel('Params N'); ax[1].set_title('Fix D=20B, vary model')
for a in ax:
    a.set_ylabel('Loss'); a.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Batch Size vs Compute Budget


In [ ]:
budget_flops = 1e20
for N in [100e6, 500e6, 1e9]:
    D = compute_optimal_tokens(N)
    total = training_flops(N, D)
    if total <= budget_flops:
        print(f"N={N/1e6:.0f}M -> D={D/1e9:.1f}B tokens fits budget ({total/budget_flops*100:.0f}% of FLOPs)")

## Summary

Applied Chinchilla scaling to estimate compute-optimal multimodal training budgets.

**Next:** Module 04 — LoRA from scratch
